# Filtered convection recovery on native Chebyshev-Gauss data

Set SPIDER_CHECKOUT to the pinned Git checkout (or install requirements-chebyshev-gauss.txt), then run prepare-numpy-files-chebyshev-gauss.ipynb first. The upstream fork is unmodified. Filtering evaluates directly on a full Gauss grid in the support-safe interior. All recovery domains span that vertical grid. See CHEBYSHEV_GAUSS_RUN.md for PACE execution, provenance and acceptance checks.

In [1]:
import gc
import os
import sys
import subprocess
from pathlib import Path

PIN = "ccc892cdc84fb183c13531a7fa654132acbe1818"
# Optional source checkout: affects only this kernel, not installed packages.
import os
os.environ["SPIDER_CHECKOUT"] = "/storage/home/hcoda1/2/ashelestov3/PySPIDER-gauss"
source_checkout = os.environ.get("SPIDER_CHECKOUT")
pyspider_checkout = None
if source_checkout:
    pyspider_checkout = Path(source_checkout).expanduser().resolve()
    expected_package = pyspider_checkout / "src" / "PySPIDER"
    if not (expected_package / "__init__.py").is_file():
        raise RuntimeError("SPIDER_CHECKOUT must point to the PySPIDER Git repository root")
    checkout_revision = subprocess.check_output(
        ["git", "-C", str(pyspider_checkout), "rev-parse", "HEAD"], text=True
    ).strip()
    checkout_changes = subprocess.check_output(
        ["git", "-C", str(pyspider_checkout), "status", "--porcelain", "--untracked-files=all", "--", "src/PySPIDER"],
        text=True,
    ).strip()
    if checkout_revision != PIN or checkout_changes:
        raise RuntimeError(f"SPIDER_CHECKOUT must contain the unmodified pinned revision {PIN}")
    loaded = sys.modules.get("PySPIDER")
    if loaded is not None and Path(loaded.__file__).resolve().parent != expected_package.resolve():
        raise RuntimeError("Restart the kernel before switching PySPIDER source checkouts")
    sys.path.insert(0, str(pyspider_checkout / "src"))

import jax
import jax.numpy as jnp
import numpy as np
from numpy.polynomial.chebyshev import chebpts1

# The quadrature fork uses the NumPy 2.x spelling. PACE's current NumPy
# provides the same routine as np.trapz, so expose a compatibility alias
# without changing either the environment or the PySPIDER package.
if not hasattr(np, "trapezoid"):
    np.trapezoid = np.trapz

try:
    from PySPIDER.commons.identify_models import *
    from PySPIDER.commons.integration import SUPPORTED_SCHEMES
    from PySPIDER.commons.quadrature_schemes import chebyshev_gauss_weights
    from PySPIDER.commons.sparse_reg_bf import (
        AIC, Initializer, ModelIterator, Residual, Scaler, Threshold
    )
    from PySPIDER.continuous.library import *
    from PySPIDER.continuous.process_library_terms import *
except (ImportError, ModuleNotFoundError) as exc:
    raise RuntimeError(
        "Set SPIDER_CHECKOUT to the pinned Git checkout, or install with: python -m pip install --no-deps "
        "-r requirements-chebyshev-gauss.txt"
    ) from exc

if "chebyshev-gauss" not in SUPPORTED_SCHEMES:
    raise RuntimeError("The installed PySPIDER is not the quadrature-enabled fork")

jax.config.update("jax_enable_x64", True)
print("PySPIDER quadrature schemes:", SUPPORTED_SCHEMES)
import PySPIDER
if pyspider_checkout is not None and Path(PySPIDER.__file__).resolve().parent != expected_package.resolve():
    raise RuntimeError("Imported PySPIDER does not match SPIDER_CHECKOUT; restart the kernel")
print("PySPIDER loaded from:", PySPIDER.__file__)

PySPIDER quadrature schemes: ('trapezoidal', 'truncated-cc-grid', 'truncated-cg-grid', 'clenshaw-curtis', 'chebyshev-gauss', 'moment-matching')
PySPIDER loaded from: /storage/home/hcoda1/2/ashelestov3/PySPIDER-gauss/src/PySPIDER/__init__.py


In [2]:
# Create a unique archive before loading data or constructing matrices.
import datetime
import hashlib
import importlib.metadata
import json
from pathlib import Path
import shutil
import subprocess
import uuid
import PySPIDER

PIN = "ccc892cdc84fb183c13531a7fa654132acbe1818"
root = Path.cwd().resolve()
if not (root / "COnvection-FILTER-Chebyshev-Gauss.ipynb").is_file():
    raise RuntimeError("Run this notebook from the repository root")
if pyspider_checkout is not None:
    pyspider_origin = {"mode": "git-checkout", "checkout": str(pyspider_checkout),
                       "revision": checkout_revision, "package_changes": checkout_changes}
else:
    distribution = importlib.metadata.distribution("PySPIDER_SR")
    direct_url = json.loads(distribution.read_text("direct_url.json") or "{}")
    if direct_url.get("vcs_info", {}).get("commit_id") != PIN:
        raise RuntimeError("Set SPIDER_CHECKOUT or install requirements-chebyshev-gauss.txt; revision differs")
    installed_package = Path(distribution.locate_file("PySPIDER")).resolve()
    if Path(PySPIDER.__file__).resolve().parent != installed_package:
        raise RuntimeError("Imported PySPIDER differs from the pinned distribution; set SPIDER_CHECKOUT explicitly")
    pyspider_origin = {"mode": "pip-distribution", "direct_url": direct_url}
run_dir = Path(os.environ.get("SPIDER_ARCHIVES", "run-archives")) / (
    datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    + "-" + uuid.uuid4().hex
)
run_dir.mkdir(parents=True, exist_ok=False)
print("RUN ARCHIVE:", run_dir.resolve(), flush=True)

def serial(value):
    if isinstance(value, np.ndarray):
        return serial(value.tolist())
    if isinstance(value, np.generic):
        return serial(value.item())
    if isinstance(value, float) and not np.isfinite(value):
        return str(value)
    if value is None or isinstance(value, (str, int, float, bool)):
        return value
    if isinstance(value, dict):
        return {str(k): serial(v) for k, v in value.items()}
    if isinstance(value, (list, tuple, set)):
        return [serial(v) for v in value]
    return str(value)

def save_json(name, value):
    # Exclusive writes preserve previous checkpoints, including on cell reruns.
    with (run_dir / name).open("x", encoding="utf-8") as stream:
        json.dump(serial(value), stream, indent=2, allow_nan=False)

def file_identity(path):
    path = Path(path).resolve()
    stat = path.stat()
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(8 * 1024 * 1024), b""):
            digest.update(block)
    return dict(path=str(path), bytes=stat.st_size, mtime_ns=stat.st_mtime_ns,
                sha256=digest.hexdigest())

snapshot_dir = run_dir / "source"
snapshot_dir.mkdir()
for name in ("COnvection-FILTER-Chebyshev-Gauss.ipynb",
             "prepare-numpy-files-chebyshev-gauss.ipynb",
             "requirements-chebyshev-gauss.txt", "CHEBYSHEV_GAUSS_RUN.md"):
    shutil.copy2(root / name, snapshot_dir / name)
shutil.copytree(root / "dedalus", snapshot_dir / "dedalus")
shutil.copytree(Path(PySPIDER.__file__).parent, snapshot_dir / "PySPIDER",
                ignore=shutil.ignore_patterns("__pycache__", "*.pyc"))
save_json("provenance.json", {
    "source_revision": subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip(),
    "working_tree": subprocess.check_output(["git", "status", "--porcelain"], text=True),
    "pyspider_revision": PIN, "pyspider_path": PySPIDER.__file__,
    "pyspider_origin": pyspider_origin,
    "packages": {d.metadata["Name"]: d.version for d in importlib.metadata.distributions()},
    "slurm_job_id": os.environ.get("SLURM_JOB_ID"),
    "model_status": "training candidates; not independently validated or thesis-selected",
    "dedalus_provenance": "Source snapshots are inferred configuration, not confirmed simulation run provenance.",
})
save_json("started.json", {"status": "started"})
# Optional externally supplied run record; user-supplied, not independently verified.
provenance_record = os.environ.get("SPIDER_RUN_PROVENANCE")
if provenance_record:
    shutil.copy2(provenance_record, run_dir / "user-supplied-run-provenance.json")
    save_json("run-provenance-identity.json", file_identity(provenance_record))

RUN ARCHIVE: /storage/home/hcoda1/2/ashelestov3/con_cheb_native/rayleigh-benard/run-archives/20261016T180743Z-aea6eec741754ec694707941355a8cc9


In [3]:
archive_path = os.environ.get("SPIDER_DATA", "/storage/scratch1/2/ashelestov3/preprocessed_paper_data_gauss.npz")
print("Loading", archive_path)

with np.load(archive_path) as data:
    # Reverse the historical archive-name swap made by preprocessing.
    V = np.asarray(data["U"])
    U = np.asarray(data["V"])
    P = np.asarray(data["P"])
    B = np.asarray(data["B"])
    x_full = np.asarray(data["x"])
    y_full = np.asarray(data["y"])
    t = np.asarray(data["t"])
    grid_kind = str(data["z_grid_kind"])
    z_interval = np.asarray(data["z_interval"], dtype=float)

assert grid_kind == "chebyshev_gauss"
assert U.shape == V.shape == P.shape == B.shape
assert U.shape == (x_full.size, y_full.size, t.size)
assert np.all(np.diff(x_full) > 0)
assert np.all(np.diff(y_full) > 0)
assert np.all(np.diff(t) > 0)

expected_y = (
    0.5 * z_interval.sum()
    + 0.5 * np.diff(z_interval)[0] * chebpts1(y_full.size)
)
np.testing.assert_allclose(y_full, expected_y, rtol=0.0, atol=5e-12)

dx_steps = np.diff(x_full)
dt_steps = np.diff(t)
dx = float(np.mean(dx_steps))
dt = float(np.mean(dt_steps))
if not np.allclose(dx_steps, dx, rtol=1e-10, atol=1e-13):
    raise ValueError("The x grid is not uniform")
if not np.allclose(dt_steps, dt, rtol=1e-8, atol=1e-12):
    raise ValueError("The current PySPIDER test expects uniformly sampled times")

Rayleigh = 1e8
Prandtl = 1.0
kappa = (Rayleigh * Prandtl) ** -0.5
nu = (Rayleigh / Prandtl) ** -0.5
print("Shape:", U.shape, "dx:", dx, "dt from np.diff(t):", dt)
print("Nonuniform z spacing range:", np.diff(y_full).min(), np.diff(y_full).max())
save_json("data.json", {"archive": file_identity(archive_path),
    "grid_kind": grid_kind, "z_interval": z_interval,
    "shape": U.shape, "time_range": [t[0], t[-1]],
    "Rayleigh": Rayleigh, "Prandtl": Prandtl, "nu": nu, "kappa": kappa,
    "parameter_status": "assumed from repository source; confirm against production run"})

Loading /storage/scratch1/2/ashelestov3/preprocessed_paper_data_gauss.npz
Shape: (4096, 1536, 101) dx: 0.0009765625 dt from np.diff(t): 2.4038999999996535e-05
Nonuniform z spacing range: 2.0916400060855267e-06 0.0010226536803383257


In [4]:
from functools import partial

def _uniform_gaussian_kernel(spacing, delta, alpha):
    sigma = delta / np.sqrt(12.0)
    radius = alpha * delta
    radius_points = int(np.ceil(radius / spacing))
    offsets = np.arange(-radius_points, radius_points + 1) * spacing
    kernel = np.exp(-0.5 * (offsets / sigma) ** 2)
    return jnp.asarray(kernel / kernel.sum())

@partial(jax.jit, static_argnums=(2,))
def _periodic_convolve_1d(array, kernel, axis):
    radius_points = kernel.size // 2
    pad = [(0, 0)] * array.ndim
    pad[axis] = (radius_points, radius_points)
    padded = jnp.pad(array, pad, mode="wrap")
    moved = jnp.moveaxis(padded, axis, 0)
    flat = moved.reshape((moved.shape[0], -1)).T
    convolved = jax.vmap(lambda line: jnp.convolve(line, kernel, mode="valid"))(flat)
    restored = convolved.T.reshape((array.shape[axis],) + moved.shape[1:])
    return jnp.moveaxis(restored, 0, axis)

def _gauss_gaussian_operator(source_coordinates, target_coordinates, delta, alpha):
    """Finite-support Gaussian convolution from Gauss source to target nodes."""
    source_coordinates = np.asarray(source_coordinates, dtype=float)
    target_coordinates = np.asarray(target_coordinates, dtype=float)
    sigma = delta / np.sqrt(12.0)
    radius = alpha * delta
    quadrature = chebyshev_gauss_weights(
        source_coordinates.size,
        float(z_interval[0]),
        float(z_interval[1]),
    )
    distances = target_coordinates[:, None] - source_coordinates[None, :]
    kernel = np.exp(-0.5 * (distances / sigma) ** 2)
    kernel[np.abs(distances) > radius] = 0.0
    weighted = kernel * quadrature[None, :]
    normalization = weighted.sum(axis=1, keepdims=True)
    if np.any(normalization <= 0):
        raise ValueError("A vertical Gaussian stencil has zero quadrature mass")
    return jnp.asarray(weighted / normalization)

delta = float(os.environ.get("SPIDER_FILTER_DX", "50")) * dx
if not np.isfinite(delta) or delta <= 0:
    raise ValueError('Filter width must be finite and positive')
alpha = 3.0
support_radius = alpha * delta
x_kernel = _uniform_gaussian_kernel(dx, delta, alpha)

safe_z_min = float(z_interval[0]) + support_radius
safe_z_max = float(z_interval[1]) - support_radius
target_nz = int(np.count_nonzero((y_full >= safe_z_min) & (y_full <= safe_z_max)))
if target_nz < 2 or safe_z_min >= safe_z_max:
    raise ValueError("The requested Gaussian support removes the entire z domain")

# Evaluate the filtered fields directly on a complete mapped Gauss grid in
# the support-safe interior. This avoids a truncated-grid moment-matching solve.
y = 0.5 * (safe_z_min + safe_z_max) + 0.5 * (safe_z_max - safe_z_min) * chebpts1(target_nz)
z_operator = _gauss_gaussian_operator(y_full, y, delta, alpha)
x = x_full.copy()  # periodic x is never trimmed

@jax.jit
def _filter_field(field):
    x_filtered = _periodic_convolve_1d(field, x_kernel, axis=0)
    return jnp.einsum("ij,xjt->xit", z_operator, x_filtered, optimize=True)

def filter_to_gauss_interior(field, label):
    print("Filtering", label)
    filtered = np.asarray(_filter_field(jnp.asarray(field)))
    gc.collect()
    return filtered

print("Filter width:", delta, "support radius:", support_radius)
print("Target Gauss z points:", y.size, "range:", (y[0], y[-1]))
# PACE-side quadrature/filter checks, before allocating filtered fields.
if delta <= 0:
    raise ValueError("Filter width must be positive")
q_source = chebyshev_gauss_weights(y_full.size, *z_interval)
for degree in range(5):
    expected = (z_interval[1]**(degree+1) - z_interval[0]**(degree+1)) / (degree+1)
    np.testing.assert_allclose(q_source @ (y_full**degree), expected, rtol=1e-11, atol=1e-12)
operator = np.asarray(z_operator)
np.testing.assert_allclose(operator.sum(axis=1), 1, rtol=1e-12, atol=1e-12)
filter_checks = {
    "constant_error": float(np.max(np.abs(operator.sum(axis=1)-1))),
    "linear_error": float(np.max(np.abs(operator @ y_full-y))),
    "second_moment_error": float(np.max(np.abs(operator @ (y_full**2)-y**2-delta**2/12))),
}
print("Filter moment diagnostics:", filter_checks)
save_json("filter-checks.json", filter_checks)

Filter width: 0.048828125 support radius: 0.146484375
Target Gauss z points: 768 range: (0.14648511442781065, 0.8535148855721894)
Filter moment diagnostics: {'constant_error': 5.551115123125783e-16, 'linear_error': 4.440892098500626e-16, 'second_moment_error': 4.070908107139948e-16}


In [5]:
# Primary filtered fields.
u_filt = filter_to_gauss_interior(U, "u")
v_filt = filter_to_gauss_interior(V, "v")
p_filt = filter_to_gauss_interior(P, "p")  # ordinary filtered pressure
b_filt = filter_to_gauss_interior(B, "b")

# Full subfilter stress tau_ij = overline(u_i u_j) - ubar_i ubar_j.
tau_xx = filter_to_gauss_interior(U * U, "u*u") - u_filt * u_filt
tau_zz = filter_to_gauss_interior(V * V, "v*v") - v_filt * v_filt
tau_xz = filter_to_gauss_interior(U * V, "u*v") - u_filt * v_filt

u_bar = np.stack((u_filt, v_filt), axis=-1)
tau = np.empty(u_filt.shape + (2, 2), dtype=u_filt.dtype)
tau[..., 0, 0] = tau_xx
tau[..., 0, 1] = tau_xz
tau[..., 1, 0] = tau_xz
tau[..., 1, 1] = tau_zz

# Subfilter buoyancy flux phi_i = overline(u_i b) - ubar_i bbar.
phi_x = filter_to_gauss_interior(U * B, "u*b") - u_filt * b_filt
phi_z = filter_to_gauss_interior(V * B, "v*b") - v_filt * b_filt
phi = np.stack((phi_x, phi_z), axis=-1)

del U, V, P, B, u_filt, v_filt, tau_xx, tau_zz, tau_xz, phi_x, phi_z
gc.collect()

print("u_bar:", u_bar.shape)
print("p, b:", p_filt.shape, b_filt.shape)
print("tau:", tau.shape, "phi:", phi.shape)

Filtering u
Filtering v
Filtering p
Filtering b
Filtering u*u
Filtering v*v
Filtering u*v
Filtering u*b
Filtering v*b
u_bar: (4096, 768, 101, 2)
p, b: (4096, 768, 101) (4096, 768, 101)
tau: (4096, 768, 101, 2, 2) phi: (4096, 768, 101, 2)


In [6]:
# Build the dataset with the fork's public nonuniform-grid and quadrature API.
unit_z = np.stack((np.zeros_like(b_filt), np.ones_like(b_filt)), axis=-1)

uobs = Observable(string="u", rank=1)
pobs = Observable(string="p", rank=0)
bobs = Observable(string="b", rank=0)
zobs = Observable(string="z̄", rank=1)
Tobs = Observable(string="T", rank=2, can_commute_indices=True, antisymmetric=False)
fobs = Observable(string="ϕ", rank=1)
observables = [uobs, bobs, pobs, zobs, Tobs, fobs]  # vorticity w intentionally omitted

data_dict = {
    "u": u_bar,
    "b": b_filt,
    "p": p_filt,
    "z̄": unit_z,
    "T": tau,
    "ϕ": phi,
}

world_size = np.asarray(b_filt.shape, dtype=int)
quadrature = {
    1: {
        "chebyshev-gauss": {
            "nodes": y,
            "interval": [safe_z_min, safe_z_max],
        }
    }
}

seed = int(os.environ.get("SPIDER_SEED", "1"))
np.random.seed(seed)
srd = SRDataset(
    world_size=world_size,
    data_dict=data_dict,
    observables=observables,
    dxs=[dx, float(np.mean(np.diff(y))), dt],
    grids=[x, y, t],
    schemes_and_options=quadrature,
    irreps=SRDataset.all_rank2_irreps(),
)
srd.make_libraries(max_complexity=3)

# Remove only actual zero/duplicate identities of the constant unit vector.
# dt(b) * z is generally nonzero and must remain a candidate term.
remove_terms = {
    "z̄_β · ∂α z̄_β",
    "z̄_α · z̄_β · z̄_β",
    "z̄_β · z̄_β · ϕ_α",
    "∂α z̄_β",
}
for irrep in srd.irreps:
    lib = srd.libs[irrep]
    before = len(lib.terms)
    lib.terms = [term for term in lib.terms if str(term) not in remove_terms]
    print(irrep, "removed", before - len(lib.terms), "symbolic terms;", len(lib.terms), "remain")

Rank 0 removed 0 symbolic terms; 68 remain
Rank 1 removed 3 symbolic terms; 100 remain
Antisymmetric rank 2 removed 1 symbolic terms; 35 remain
Symmetric trace-free rank 2 removed 1 symbolic terms; 50 remain


In [8]:
# Test-run defaults. Increase SPIDER_TEST_NDOMAINS after validating the path.
dom_x = 64
dom_z = 64  # fixed nodes require every domain to span the target Gauss grid
dom_time = 40
ndomains = 32 #int(os.environ.get("SPIDER_TEST_NDOMAINS", "256"))

if ndomains < 1 or dom_x > world_size[0] or dom_time > world_size[2]:
    raise ValueError("Requested test domain is larger than the available data")

# Forking after JAX has initialized worker threads is unsafe. Serial evaluation
# is therefore the test default. Set SPIDER_TEST_PARALLEL=1 only in a process
# where JAX filtering and PySPIDER recovery have been separated safely.
parallel = os.environ.get("SPIDER_TEST_PARALLEL", "0") == "1"
available_cores = os.cpu_count() or 1
num_processors = min(ndomains, max(1, available_cores - 4)) if parallel else 1

quadrature = {1: {"truncated-cg-grid": {"degree": 16}}}
srd.schemes_and_options = quadrature

srd.make_domains(
    ndomains=ndomains,
    domain_size=[dom_x, dom_z, dom_time],
    pad=0,
)
srd.make_weights(m=16, qmax=0)

domain = srd.domains[0]
z_lo = domain.min_corner[1]
z_hi = domain.max_corner[1]
np.testing.assert_allclose(
    srd.weight_dxs[1], (y[z_hi] - y[z_lo])/2, rtol=1e-12
)


L = (dom_x - 1) * dx
T = (dom_time - 1) * dt  # no obsolete factor of 10
srd.set_LT_scale(L=L, T=T)
print(
    "Domains:", ndomains,
    "shape:", srd.domain_size,
    "mode:", "parallel" if parallel else "serial",
    "workers:", num_processors,
)
print("Characteristic L, T:", L, T)
srd.make_library_matrices(
    debug=False,
    parallel=parallel,
    num_processors=num_processors if parallel else None,
)

Domains: 32 shape: [64, 64, 40] mode: serial workers: 1
Characteristic L, T: 0.0615234375 0.0009375209999998649


In [9]:
# Evaluate the known balance on the original Q BEFORE numerical pruning.
# A diagnostic term may be small or proportional to another sampled column;
# that must not prevent us from evaluating the prescribed physical equation.
# dt(u) + u.grad(u) + grad(p) - nu*lap(u) - b*z + div(tau) = 0.
momentum_coefficients = {
    "∂t u_α": 1.0,
    "u_β · ∂β u_α": 1.0,
    "∂α p": 1.0,
    "∂β² u_α": -nu,
    "b · z̄_α": -1.0,
    "∂β T_αβ": 1.0,
}

vector_irrep = next(irrep for irrep in srd.irreps if str(irrep) == "Rank 1")
vector_lib = srd.libs[vector_irrep]
term_index = {str(term): index for index, term in enumerate(vector_lib.terms)}
missing = [name for name in momentum_coefficients if name not in term_index]
if missing:
    raise KeyError(f"Known momentum terms missing from the original library: {missing}")

known_columns = []
known_residual = np.zeros(vector_lib.Q.shape[0])
for name, coefficient in momentum_coefficients.items():
    contribution = coefficient * vector_lib.Q[:, term_index[name]]
    known_columns.append(contribution)
    known_residual += contribution

known_denominator = sum(np.linalg.norm(column) for column in known_columns)
if not np.isfinite(known_denominator) or known_denominator <= 0:
    raise ValueError("Known momentum contributions have zero or nonfinite norm")
known_relative_residual = np.linalg.norm(known_residual) / known_denominator
print("Known filtered momentum relative balance residual:", known_relative_residual)
print("Known coefficients:", momentum_coefficients)
for name, contribution in zip(momentum_coefficients, known_columns):
    print(f"  {name}: contribution norm = {np.linalg.norm(contribution):.6e}")

Known filtered momentum relative balance residual: 2.939882222393091e-06
Known coefficients: {'∂t u_α': 1.0, 'u_β · ∂β u_α': 1.0, '∂α p': 1.0, '∂β² u_α': -0.0001, 'b · z̄_α': -1.0, '∂β T_αβ': 1.0}
  ∂t u_α: contribution norm = 3.481516e+00
  u_β · ∂β u_α: contribution norm = 8.633099e+00
  ∂α p: contribution norm = 1.026456e+01
  ∂β² u_α: contribution norm = 3.930044e-02
  b · z̄_α: contribution norm = 4.697013e+00
  ∂β T_αβ: contribution norm = 1.356754e-01


In [10]:
# Repeated analysis gets its own archive; never overwrite earlier checkpoints.
def fork_analysis_archive(include_matrices=False):
    global run_dir
    parent = run_dir
    new_dir = parent.parent / (
        datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%SZ")
        + "-" + uuid.uuid4().hex
    )
    new_dir.mkdir(parents=True, exist_ok=False)
    names = ["provenance.json", "data.json", "filter-checks.json",
             "user-supplied-run-provenance.json", "run-provenance-identity.json"]
    if include_matrices:
        names += [p.name for pattern in ("matrix-*.npz", "terms-*.json", "known-momentum*.json",
                                        "known-momentum*.npz", "matrix-diagnostics.json",
                                        "pruning.json", "settings.json") for p in parent.glob(pattern)]
    for name in dict.fromkeys(names):
        if (parent / name).is_file():
            shutil.copy2(parent / name, new_dir / name)
    if (parent / "source").is_dir():
        shutil.copytree(parent / "source", new_dir / "source")
    # Keep inherited input-source snapshots and the currently saved notebook distinct.
    shutil.copy2(root / "COnvection-FILTER-Chebyshev-Gauss.ipynb", new_dir / "resumed-notebook.ipynb")
    run_dir = new_dir
    save_json("resumed.json", {
        "parent_archive": str(parent.resolve()),
        "reused_matrices": include_matrices,
        "source_revision_at_resume": subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip(),
        "source_note": "source/ is inherited input provenance; resumed-notebook.ipynb is the saved notebook at resume, not proof of executed cell contents",
    })
    print("NEW ANALYSIS ARCHIVE:", run_dir.resolve(), flush=True)

def save_npz(name, **arrays):
    with (run_dir / name).open("xb") as stream:
        np.savez_compressed(stream, **arrays)
def prune_numerical_library_columns(lib, zero_rtol=1e-12, duplicate_rtol=1e-10):
    """Prune scaled numerical zeros and proportional columns after diagnostics."""
    Q = np.asarray(lib.Q)
    if Q.ndim != 2 or Q.shape[1] != len(lib.terms) or not np.all(np.isfinite(Q)):
        raise ValueError(f"{lib.irrep}: invalid or nonfinite library matrix")
    # Use the same characteristic sizes as PySPIDER Scaler. Raw column
    # magnitudes are not comparable across fields and derivative orders.
    sizes = np.asarray(lib.col_weights, dtype=float)
    if sizes.shape != (Q.shape[1],) or np.any(~np.isfinite(sizes)) or np.any(sizes <= 0):
        raise ValueError(f"{lib.irrep}: invalid characteristic column sizes")
    scaled_Q = Q / sizes[None, :]
    if not np.all(np.isfinite(scaled_Q)):
        raise ValueError(f"{lib.irrep}: nonfinite scaled library matrix")
    norms = np.linalg.norm(scaled_Q, axis=0)
    scale = norms.max(initial=0.0)
    zero = norms <= zero_rtol * scale

    keep = []
    duplicates = []
    normalized = np.zeros_like(Q, dtype=float)
    nonzero = ~zero
    normalized[:, nonzero] = scaled_Q[:, nonzero] / norms[nonzero]

    # Each vector has unit norm, so duplicate_rtol is already relative.
    tolerance = duplicate_rtol
    for column in np.flatnonzero(nonzero):
        vector = normalized[:, column]
        duplicate_of = None
        for previous in keep:
            if min(
                np.linalg.norm(vector - normalized[:, previous]),
                np.linalg.norm(vector + normalized[:, previous]),
            ) <= tolerance:
                duplicate_of = previous
                break
        if duplicate_of is None:
            keep.append(column)
        else:
            duplicates.append((column, duplicate_of))

    keep = np.asarray(keep, dtype=int)
    old_terms = list(lib.terms)
    lib.Q = Q[:, keep]
    lib.terms = [old_terms[index] for index in keep]
    if lib.col_weights is not None:
        weights = np.asarray(lib.col_weights)
        lib.col_weights = list(weights[keep])

    print(
        lib.irrep,
        "removed", int(zero.sum()), "zero and", len(duplicates),
        "duplicate columns;", len(keep), "remain",
    )
    return {
        "zero_terms": [old_terms[index] for index in np.flatnonzero(zero)],
        "duplicate_terms": [
            (old_terms[index], old_terms[previous]) for index, previous in duplicates
        ],
    }

# Check for previous/partial attempts BEFORE writing any matrix.
if any(run_dir.glob("matrix-original-*.npz")) or any(run_dir.glob("terms-original-*.json")):
    fork_analysis_archive()

# Restore the pre-pruning input on a repeated analysis of the same matrices.
# A newly constructed Q (different object) starts a new input snapshot.
previous = globals().get("_pruning_input_snapshot")
if previous is not None and previous["dataset"] is srd and all(
    srd.libs[irrep].Q is previous["outputs"].get(irrep) for irrep in srd.irreps
):
    for irrep, (Q, terms, weights) in previous["inputs"].items():
        srd.libs[irrep].Q = Q
        srd.libs[irrep].terms = list(terms)
        srd.libs[irrep].col_weights = list(weights)
_pruning_input_snapshot = {
    "dataset": srd,
    "inputs": {irrep: (lib.Q, list(lib.terms), list(lib.col_weights)) for irrep, lib in srd.libs.items()},
    "outputs": {},
}
# "original" means the input to this pruning pass. On legacy kernels that
# already pruned without a snapshot, earlier removed terms remain in the parent archive.
# Preserve every input column before mutating libraries.
for i, irrep in enumerate(srd.irreps):
    lib = srd.libs[irrep]
    save_npz(f"matrix-original-{i}.npz", Q=lib.Q,
                        col_weights=lib.col_weights)
    save_json(f"terms-original-{i}.json", {"irrep": str(irrep), "terms": lib.terms})
save_json("known-momentum.json", {"coefficients": momentum_coefficients,
    "relative_residual": known_relative_residual,
    "contribution_norms": dict(zip(momentum_coefficients, map(np.linalg.norm, known_columns)))})

pruning_report = {
    irrep: prune_numerical_library_columns(srd.libs[irrep])
    for irrep in srd.irreps
}
_pruning_input_snapshot["outputs"] = {irrep: lib.Q for irrep, lib in srd.libs.items()}
matrix_diagnostics = {}
for i, irrep in enumerate(srd.irreps):
    lib = srd.libs[irrep]
    scaled = lib.Q / np.asarray(lib.col_weights)[None, :]
    singular = np.linalg.svd(scaled, compute_uv=False)
    rank = int(np.sum(singular > singular[0] * max(scaled.shape) * np.finfo(float).eps)) if singular.size else 0
    matrix_diagnostics[str(irrep)] = dict(shape=scaled.shape, rank=rank, singular_values=singular)
    print("Matrix diagnostics:", irrep, scaled.shape, "numerical rank:", rank)
    save_npz(f"matrix-pruned-{i}.npz", Q=lib.Q, col_weights=lib.col_weights)
    save_json(f"terms-pruned-{i}.json", {"irrep": str(irrep), "terms": lib.terms})
save_json("matrix-diagnostics.json", matrix_diagnostics)
save_json("pruning.json", pruning_report)
save_json("settings.json", {
    "baseline": "prescribed filtered momentum with full tau and ordinary pressure; no fitted closure baseline",
    "filter": {"delta": delta, "alpha": alpha, "x_boundary": "periodic", "z": "Fejer-I weighted Gaussian", "checks": filter_checks},
    "library": {"max_complexity": 3, "observables": observables, "remove_terms": remove_terms,
                "zero_rtol": 1e-12, "duplicate_rtol": 1e-10},
    "integration": {"quadrature": quadrature, "dx": dx, "dt": dt, "L": L, "T": T,
                    "domain_size": srd.domain_size, "weight_m": 16, "weight_qmax": 0,
                    "parallel": parallel, "num_processors": num_processors},
    "domains": [{"min": d.min_corner, "max": d.max_corner} for d in srd.domains],
    "seed": seed, "x": x, "y": y, "t": t,
    "validation": "Random row split only; overlapping domains are not independent validation",
})
save_npz('known-momentum-arrays.npz', residual=known_residual, contributions=np.stack(known_columns))


Rank 0 removed 9 zero and 2 duplicate columns; 57 remain
Rank 1 removed 11 zero and 1 duplicate columns; 88 remain
Antisymmetric rank 2 removed 6 zero and 0 duplicate columns; 29 remain
Symmetric trace-free rank 2 removed 6 zero and 0 duplicate columns; 44 remain
Matrix diagnostics: Rank 0 (32, 57) numerical rank: 32
Matrix diagnostics: Rank 1 (64, 88) numerical rank: 64
Matrix diagnostics: Antisymmetric rank 2 (32, 29) numerical rank: 29
Matrix diagnostics: Symmetric trace-free rank 2 (64, 44) numerical rank: 44


In [11]:
# Rerunning regression reuses the archived matrices in a fresh attempt directory.
if (run_dir / "regression-settings.json").exists():
    fork_analysis_archive(include_matrices=True)
# Sparse recovery follows only after the direct known-balance check.
reg_opts_list = []
for irrep in srd.irreps:
    lib = srd.libs[irrep]
    if os.environ.get("SPIDER_DIAGNOSTIC_ONLY", "1") != "1" and (len(lib.terms) == 0 or int(0.8 * lib.Q.shape[0]) <= min(10, len(lib.terms))):
        raise ValueError(f'{irrep}: empty library or too few training rows; increase SPIDER_TEST_NDOMAINS')
    scaler = Scaler(
        sub_inds=None,
        char_sizes=lib.col_weights,
        row_norms=None,
        unit_rows=True,
        train_fraction=0.8,
    )
    reg_opts_list.append({
        "scaler": scaler,
        "initializer": Initializer(method="power", start_k=min(10, len(lib.terms))),
        "residual": Residual(residual_type="matrix_relative"),
        "model_iterator": ModelIterator(max_k=min(10, len(lib.terms)), backward_forward=True, max_passes=3),
        "threshold": Threshold(threshold_type="jump", gamma=1.5, delta=1e-8, n_terms=None),
        "verbose": False,
        "seed": seed,
        "inhomog": False,
        "inhomog_col": None,
    })

# interleave_identify returns accepted equations only. Capture EVERY completed
# sparse-regression call (including rejected candidates) without editing PySPIDER.
import importlib
import random
import traceback
identify_module = importlib.import_module("PySPIDER.commons.identify_models")
original_regression = identify_module.sparse_reg_bf
attempts = []
save_json("regression-settings.json", {
    "options": [{k: vars(v) if hasattr(v, "__dict__") else v for k, v in opts.items()}
                for opts in reg_opts_list],
    "threshold": 1e-6,
    "diagnostic_only": os.environ.get("SPIDER_DIAGNOSTIC_ONLY", "1") == "1",
    "candidate_errors": "all_lambdas/all_lambdas_test are upstream internal scaled errors; lambd/lambda_test are postprocessed. Do not compare them as identical normalizations.",
})

def capture_regression(Q, **options):
    number = len(attempts)
    selected = options["scaler"].sub_inds
    full_lib = next(lib for lib in srd.libs.values() if lib.Q is Q)
    attempt = {"number": number, "irrep": str(full_lib.irrep),
               "full_terms": full_lib.terms, "selected_indices": selected,
               "selected_terms": options.get("term_names"),
               "options": {k: vars(v) if hasattr(v, "__dict__") else v
                           for k, v in options.items()}}
    attempts.append(attempt)
    # Match upstream's seeded random row split for exact reproducibility.
    rows = Q.shape[0]
    train = random.Random(options["seed"]).sample(range(rows), int(rows * 0.8))
    attempt["train_rows"] = train
    attempt["test_rows"] = sorted(set(range(rows)) - set(train))
    save_json(f"attempt-{number:04d}-input.json", attempt)
    try:
        if np.any(np.linalg.norm(Q, axis=1) == 0):
            raise ValueError("Zero full-library row cannot be normalized; inspect archived Q")
        result = original_regression(Q, **options)
        result_data = dict(vars(result))
        # xi uses full-library physical coefficients; upstream all_xis stores
        # its internal scaled sublibrary candidates. Preserve both conventions.
        all_xis = result.all_xis
        if all_xis is not None:
            all_xis = np.asarray(all_xis)
            sizes = np.asarray(full_lib.col_weights)[selected]
            result_data["all_xis_physical_sublibrary"] = all_xis / sizes[None, :]
        result_data["coefficient_conventions"] = {
            "xi": "physical coefficients indexed by full_terms",
            "all_xis": "upstream scaled coefficients indexed by selected_terms",
            "all_xis_physical_sublibrary": "all_xis divided by selected characteristic sizes; arbitrary homogeneous normalization",
        }
        save_json(f"attempt-{number:04d}-result.json", result_data)
        return result
    except Exception:
        save_json(f"attempt-{number:04d}-failure.json", {"traceback": traceback.format_exc()})
        raise

identify_module.sparse_reg_bf = capture_regression
try:
    if os.environ.get("SPIDER_DIAGNOSTIC_ONLY", "1") == "1":
        save_json("complete.json", {"status": "diagnostics complete; regression not requested"})
        print("Diagnostics archived. Set SPIDER_DIAGNOSTIC_ONLY=0 for candidate recovery.")
    else:
        eqs, lambdas, reg_results, derived_eqs, excluded_terms = interleave_identify(
            [srd.libs[irrep] for irrep in srd.irreps], reg_opts_list,
            threshold=1e-6, report_accuracy=True,
        )
        save_json("accepted.json", {
            "equations": [{"equation": str(eq), "terms": eq.terms, "coefficients": eq.coeffs}
                          for eq in eqs],
            "errors": lambdas, "derived_equations": derived_eqs, "excluded_terms": excluded_terms,
            "status": "training candidates only", "thesis_selected": [],
            "acceptance_threshold": 1e-6,
        })
        save_json("complete.json", {"status": "candidate recovery complete", "attempts": len(attempts)})
except Exception:
    save_json("failed.json", {"traceback": traceback.format_exc(), "completed_or_failed_attempts": len(attempts)})
    raise
finally:
    identify_module.sparse_reg_bf = original_regression

Diagnostics archived. Set SPIDER_DIAGNOSTIC_ONLY=0 for candidate recovery.
